# 01 — Chest X-ray Data Preparation

**Academic prototype only.** This notebook is part of a university final project.
It is **not** a clinical diagnostic system, is **not** validated for patient care,
and must **never** be used to diagnose, screen, or triage real patients.

## Why this notebook exists

The original Coursera / NIH course validation split has **two** defects, not one:

1. **Patient leakage** — 197 of the 199 `PatientId`s in `valid-small.csv` also
   appear in `train-small.csv`.
2. **Image duplication** — 198 of the 200 rows in `valid-small.csv` are
   byte-identical copies of rows in `train-small.csv`.

Fixing only (1) leaves the duplicates in place, because a patient-level split
keeps both copies of a duplicated image on the same side. The result looks clean
to an overlap check while still inflating every count.

**Our approach:**
- Keep the official `test.csv` **unchanged** for final evaluation only.
- Combine only `train-small.csv` + `valid-small.csv`.
- **Deduplicate by `Image` first**, after proving the duplicates are exact copies.
- Then rebuild a **patient-level** train/validation split with `GroupShuffleSplit`
  so no patient — and no image — appears in both clean splits.

Raw files under `data/raw/` and `data_dl/` are never modified.

## 0. Setup and paths

All paths are relative to the **project root**.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "xray" / "nih"
IMAGE_DIR = RAW_DIR / "images-small"
TRAIN_CSV = RAW_DIR / "train-small.csv"
VALID_CSV = RAW_DIR / "valid-small.csv"
TEST_CSV = RAW_DIR / "test.csv"

OUT_DIR = PROJECT_ROOT / "data" / "processed" / "xray"
TRAIN_CLEAN = OUT_DIR / "train_clean.csv"
VALID_CLEAN = OUT_DIR / "valid_clean.csv"

LABEL_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Effusion",
    "Emphysema", "Fibrosis", "Hernia", "Infiltration", "Mass", "Nodule",
    "Pleural_Thickening", "Pneumonia", "Pneumothorax",
]

print("Project root:", PROJECT_ROOT)
print("Image dir   :", IMAGE_DIR)
print("Train CSV   :", TRAIN_CSV)
print("Valid CSV   :", VALID_CSV)
print("Test CSV    :", TEST_CSV)
print("Output dir  :", OUT_DIR)

Project root: D:\AI Engineering\medicaldiagnostic_finalproject
Image dir   : D:\AI Engineering\medicaldiagnostic_finalproject\data\raw\xray\nih\images-small
Train CSV   : D:\AI Engineering\medicaldiagnostic_finalproject\data\raw\xray\nih\train-small.csv
Valid CSV   : D:\AI Engineering\medicaldiagnostic_finalproject\data\raw\xray\nih\valid-small.csv
Test CSV    : D:\AI Engineering\medicaldiagnostic_finalproject\data\raw\xray\nih\test.csv
Output dir  : D:\AI Engineering\medicaldiagnostic_finalproject\data\processed\xray


In [2]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

print("pandas:", pd.__version__)
import sklearn
print("sklearn:", sklearn.__version__)

pandas: 3.0.5
sklearn: 1.9.0


## 1. Load and validate the three CSV files

We expect 14 disease-label columns plus `Image` and `PatientId`.
Every referenced filename must exist under `images-small/`.
Hidden macOS files (`._*`, `.DS_Store`) are ignored when scanning the folder.

In [3]:
def load_split_csv(path: Path, name: str) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f"Missing {name}: {path}")
    df = pd.read_csv(path)
    required = ["Image", "PatientId"] + LABEL_COLS
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")
    # Keep a stable column order for later saves
    return df[required].copy()


train_raw = load_split_csv(TRAIN_CSV, "train-small.csv")
valid_raw = load_split_csv(VALID_CSV, "valid-small.csv")
test_raw = load_split_csv(TEST_CSV, "test.csv")

print("Confirmed 14 disease-label columns:")
print(LABEL_COLS)
print()
for name, df in [("train-small", train_raw), ("valid-small", valid_raw), ("test", test_raw)]:
    print(f"{name:12s}  rows={len(df):4d}  unique patients={df['PatientId'].nunique():4d}")

Confirmed 14 disease-label columns:
['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']

train-small   rows=1000  unique patients= 928
valid-small   rows= 200  unique patients= 199
test          rows= 420  unique patients= 389


In [4]:
# Real PNGs only — exclude macOS AppleDouble / Finder junk
available_images = {
    p.name
    for p in IMAGE_DIR.iterdir()
    if p.is_file()
    and p.suffix.lower() == ".png"
    and not p.name.startswith("._")
    and p.name != ".DS_Store"
}
print(f"PNG files in images-small/ (excluding ._* / .DS_Store): {len(available_images)}")


def assert_images_exist(df: pd.DataFrame, name: str) -> None:
    missing = sorted(set(df["Image"]) - available_images)
    if missing:
        preview = ", ".join(missing[:5])
        raise FileNotFoundError(
            f"{name}: {len(missing)} referenced image(s) missing. Examples: {preview}"
        )
    print(f"[ok] {name}: all {len(df)} image paths exist")


assert_images_exist(train_raw, "train-small.csv")
assert_images_exist(valid_raw, "valid-small.csv")
assert_images_exist(test_raw, "test.csv")

PNG files in images-small/ (excluding ._* / .DS_Store): 1422
[ok] train-small.csv: all 1000 image paths exist
[ok] valid-small.csv: all 200 image paths exist
[ok] test.csv: all 420 image paths exist


In [5]:
def positive_counts(df: pd.DataFrame, name: str) -> pd.DataFrame:
    counts = df[LABEL_COLS].sum().astype(int)
    out = pd.DataFrame({"positives": counts})
    out.index.name = "disease"
    print(f"\nPositive counts — {name} (n={len(df)})")
    display(out.T)
    return out


_ = positive_counts(train_raw, "train-small")
_ = positive_counts(valid_raw, "valid-small")
_ = positive_counts(test_raw, "official test")


Positive counts — train-small (n=1000)


disease,Atelectasis,Cardiomegaly,Consolidation,Edema,Effusion,Emphysema,Fibrosis,Hernia,Infiltration,Mass,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax
positives,106,20,33,16,128,13,14,2,175,45,54,21,10,38



Positive counts — valid-small (n=200)


disease,Atelectasis,Cardiomegaly,Consolidation,Edema,Effusion,Emphysema,Fibrosis,Hernia,Infiltration,Mass,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax
positives,19,4,7,2,27,3,1,1,29,9,11,6,1,10



Positive counts — official test (n=420)


disease,Atelectasis,Cardiomegaly,Consolidation,Edema,Effusion,Emphysema,Fibrosis,Hernia,Infiltration,Mass,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax
positives,60,50,53,50,53,56,61,50,59,60,54,58,50,55


## 2. Demonstrate leakage in the original course split

Patients shared between the original train and validation CSVs are the reason
we rebuild the split.

In [6]:
orig_overlap = set(train_raw["PatientId"]) & set(valid_raw["PatientId"])
print(f"PatientId overlap between original train-small and valid-small: {len(orig_overlap)}")
if len(orig_overlap):
    print("Example leaked PatientIds:", sorted(orig_overlap)[:10])

PatientId overlap between original train-small and valid-small: 197
Example leaked PatientIds: [121, 375, 443, 589, 658, 720, 918, 1168, 1212, 1232]


## 3. Build the development pool (train-small + valid-small), deduplicated

`test.csv` is **not** included here. It stays reserved for final evaluation.

The two course CSVs **overlap by image, not just by patient**: `valid-small.csv`
is almost entirely a subset of `train-small.csv`, so 198 filenames appear twice
in the concatenated pool. Removing the 197-patient overlap alone does *not* fix
this — grouping by `PatientId` sends both copies of a duplicated image to the
same side of the split, which hides the problem from any overlap check while
still inflating the row counts.

So the pool is **deduplicated by `Image` before the split**, after asserting the
duplicates are exact copies (same `PatientId`, same 14 labels) and therefore
safe to collapse.

In [7]:
# Step 1-2: concatenate the two course CSVs into the development pool.
dev_pool_raw = pd.concat([train_raw, valid_raw], ignore_index=True)
dup_records = len(dev_pool_raw) - dev_pool_raw["Image"].nunique()

# Step 4a: a duplicate is only safe to drop if every copy agrees on PatientId
# and all 14 labels. Collapse fully-identical rows, then look for any Image that
# still has more than one distinct record - that would be a genuine conflict.
KEY_COLS = ["Image", "PatientId"] + LABEL_COLS
conflicting = int(
    dev_pool_raw[KEY_COLS].drop_duplicates().groupby("Image").size().gt(1).sum()
)
assert conflicting == 0, (
    f"{conflicting} Image(s) carry conflicting PatientId/label values across "
    "duplicate rows - these cannot be deduplicated automatically."
)

# Step 3: keep exactly one copy of each Image.
dev_pool = dev_pool_raw.drop_duplicates(subset="Image", keep="first").reset_index(drop=True)

# Step 4b: one PatientId per Image.
assert dev_pool.groupby("Image")["PatientId"].nunique().max() == 1, \
    "At least one Image maps to more than one PatientId."

# Step 6: the development pool must be fully disjoint from the official test set.
assert not (set(dev_pool["Image"]) & set(test_raw["Image"])), \
    "Image overlap between development pool and official test.csv."
assert not (set(dev_pool["PatientId"]) & set(test_raw["PatientId"])), \
    "PatientId overlap between development pool and official test.csv."

print(f"Concatenated rows (train-small + valid-small) : {len(dev_pool_raw)}")
print(f"Duplicate Image records removed               : {dup_records}")
print(f"Development pool                              : "
      f"{len(dev_pool)} unique images, {dev_pool['PatientId'].nunique()} patients")
print(f"Official test rows left untouched             : {len(test_raw)}")
print()
print("[ok] duplicates were exact copies - no conflicting labels")
print("[ok] one PatientId per Image")
print("[ok] development pool shares no Image and no PatientId with test.csv")

Concatenated rows (train-small + valid-small) : 1200
Duplicate Image records removed               : 198
Development pool                              : 1002 unique images, 930 patients
Official test rows left untouched             : 420

[ok] duplicates were exact copies - no conflicting labels
[ok] one PatientId per Image
[ok] development pool shares no Image and no PatientId with test.csv


## 4. Patient-level train / validation split

Use `GroupShuffleSplit` so entire patients stay in one split:
- `groups = PatientId`
- `test_size = 0.20` (this becomes the clean validation set)
- `random_state = 42`

The splitter configuration is **unchanged** from the previous version of this
notebook — the patient partition it produces was already correct. The only
difference is that it now receives the deduplicated `dev_pool`, so each image
is assigned exactly once.

Outputs (derived data only; raw CSVs are never overwritten):
- `data/processed/xray/train_clean.csv` — 795 rows / 795 images / 744 patients
- `data/processed/xray/valid_clean.csv` — 207 rows / 207 images / 186 patients

In [8]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Step 7: unchanged splitter - only the input is now deduplicated.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, valid_idx = next(
    splitter.split(dev_pool, groups=dev_pool["PatientId"])
)

train_clean = dev_pool.iloc[train_idx].reset_index(drop=True)
valid_clean = dev_pool.iloc[valid_idx].reset_index(drop=True)

# Step 8: write only after the shape is exactly what we expect.
assert (len(train_clean), train_clean["Image"].nunique(),
        train_clean["PatientId"].nunique()) == (795, 795, 744), \
    f"train_clean shape changed: {len(train_clean)} rows"
assert (len(valid_clean), valid_clean["Image"].nunique(),
        valid_clean["PatientId"].nunique()) == (207, 207, 186), \
    f"valid_clean shape changed: {len(valid_clean)} rows"

train_clean.to_csv(TRAIN_CLEAN, index=False)
valid_clean.to_csv(VALID_CLEAN, index=False)

for path, df in [(TRAIN_CLEAN, train_clean), (VALID_CLEAN, valid_clean)]:
    print(f"Saved: {path.relative_to(PROJECT_ROOT)}")
    print(f"  rows={len(df)}  images={df['Image'].nunique()}  "
          f"patients={df['PatientId'].nunique()}")

Saved: data\processed\xray\train_clean.csv


  rows=795  images=795  patients=744
Saved: data\processed\xray\valid_clean.csv
  rows=207  images=207  patients=186


## 5. Verification

Checks (all are hard `assert`s — the notebook fails loudly rather than writing
numbers we cannot trust into the report or the app):

1. No duplicate `Image` **within** `train_clean` or **within** `valid_clean`
2. No `Image` overlap between train, validation and official test
3. No `PatientId` overlap between train, validation and official test
4. `train + validation = 1002` unique development images
5. Every cleaned row still points to an existing PNG
6. Class-positive counts and prevalence (%) for each split

In [9]:
train_images = set(train_clean["Image"])
valid_images = set(valid_clean["Image"])
test_images = set(test_raw["Image"])

train_patients = set(train_clean["PatientId"])
valid_patients = set(valid_clean["PatientId"])
test_patients = set(test_raw["PatientId"])

# 1. No duplicate Image inside either clean split.
assert len(train_clean) == len(train_images), \
    f"train_clean has {len(train_clean) - len(train_images)} duplicate Image row(s)"
assert len(valid_clean) == len(valid_images), \
    f"valid_clean has {len(valid_clean) - len(valid_images)} duplicate Image row(s)"

# 2. No Image shared between any pair of splits.
assert not (train_images & valid_images), "Image overlap: train_clean / valid_clean"
assert not (train_images & test_images), "Image overlap: train_clean / test"
assert not (valid_images & test_images), "Image overlap: valid_clean / test"

# 3. No PatientId shared between any pair of splits.
assert not (train_patients & valid_patients), "PatientId overlap: train_clean / valid_clean"
assert not (train_patients & test_patients), "PatientId overlap: train_clean / test"
assert not (valid_patients & test_patients), "PatientId overlap: valid_clean / test"

# 4. The two clean splits reconstruct the whole deduplicated development pool.
n_dev = len(train_clean) + len(valid_clean)
assert n_dev == len(dev_pool) == 1002, \
    f"Expected 1002 unique development images, got {n_dev}"

print(f"Duplicate Image rows in train_clean : {len(train_clean) - len(train_images)}")
print(f"Duplicate Image rows in valid_clean : {len(valid_clean) - len(valid_images)}")
print(f"Image overlap  train ∩ valid        : {len(train_images & valid_images)}")
print(f"Image overlap  train ∩ test         : {len(train_images & test_images)}")
print(f"Image overlap  valid ∩ test         : {len(valid_images & test_images)}")
print(f"Patient overlap train ∩ valid       : {len(train_patients & valid_patients)}")
print(f"Patient overlap train ∩ test        : {len(train_patients & test_patients)}")
print(f"Patient overlap valid ∩ test        : {len(valid_patients & test_patients)}")
print(f"train + validation images           : {n_dev}")
print()
print("[ok] no duplicate Image within either clean split")
print("[ok] no Image overlap between train / validation / test")
print("[ok] no PatientId overlap between train / validation / test")
print("[ok] train + validation = 1002 unique development images")

# 5. Every referenced PNG still exists on disk.
assert_images_exist(train_clean, "train_clean.csv")
assert_images_exist(valid_clean, "valid_clean.csv")

Duplicate Image rows in train_clean : 0
Duplicate Image rows in valid_clean : 0
Image overlap  train ∩ valid        : 0
Image overlap  train ∩ test         : 0
Image overlap  valid ∩ test         : 0
Patient overlap train ∩ valid       : 0
Patient overlap train ∩ test        : 0
Patient overlap valid ∩ test        : 0
train + validation images           : 1002

[ok] no duplicate Image within either clean split
[ok] no Image overlap between train / validation / test
[ok] no PatientId overlap between train / validation / test
[ok] train + validation = 1002 unique development images
[ok] train_clean.csv: all 795 image paths exist
[ok] valid_clean.csv: all 207 image paths exist


In [10]:
def prevalence_table(df: pd.DataFrame, name: str) -> pd.DataFrame:
    positives = df[LABEL_COLS].sum().astype(int)
    prevalence = (100.0 * positives / len(df)).round(2)
    table = pd.DataFrame({
        "positives": positives,
        "prevalence_%": prevalence,
    })
    table.index.name = "disease"
    print(f"\n{name}: n={len(df)} images, patients={df['PatientId'].nunique()}")
    display(table)
    return table


train_prev = prevalence_table(train_clean, "train_clean")
valid_prev = prevalence_table(valid_clean, "valid_clean")
test_prev = prevalence_table(test_raw, "official test (unchanged)")

print("\nHernia positives (rare class reminder):")
print(f"  train_clean : {int(train_clean['Hernia'].sum())}")
print(f"  valid_clean : {int(valid_clean['Hernia'].sum())}")
print(f"  official test: {int(test_raw['Hernia'].sum())}")


train_clean: n=795 images, patients=744


,positives,prevalence_%
disease,,
Atelectasis,89,11.19
Cardiomegaly,14,1.76
Consolidation,30,3.77
Edema,9,1.13
Effusion,109,13.71
Emphysema,12,1.51
Fibrosis,8,1.01
Hernia,2,0.25
Infiltration,140,17.61



valid_clean: n=207 images, patients=186


,positives,prevalence_%
disease,,
Atelectasis,18,8.70
Cardiomegaly,6,2.90
Consolidation,3,1.45
Edema,7,3.38
Effusion,19,9.18
Emphysema,1,0.48
Fibrosis,6,2.90
Hernia,1,0.48
Infiltration,35,16.91



official test (unchanged): n=420 images, patients=389


,positives,prevalence_%
disease,,
Atelectasis,60,14.29
Cardiomegaly,50,11.90
Consolidation,53,12.62
Edema,50,11.90
Effusion,53,12.62
Emphysema,56,13.33
Fibrosis,61,14.52
Hernia,50,11.90
Infiltration,59,14.05



Hernia positives (rare class reminder):
  train_clean : 2
  valid_clean : 1
  official test: 50


In [11]:
summary = pd.DataFrame([
    {"split": "train_clean", "rows": len(train_clean),
     "unique_images": train_clean["Image"].nunique(),
     "unique_patients": train_clean["PatientId"].nunique()},
    {"split": "valid_clean", "rows": len(valid_clean),
     "unique_images": valid_clean["Image"].nunique(),
     "unique_patients": valid_clean["PatientId"].nunique()},
    {"split": "test (untouched)", "rows": len(test_raw),
     "unique_images": test_raw["Image"].nunique(),
     "unique_patients": test_raw["PatientId"].nunique()},
])
rows_equal_images = bool((summary["rows"] == summary["unique_images"]).all())
all_patients = set(dev_pool["PatientId"]) | set(test_raw["PatientId"])

print("=" * 64)
print("INTEGRITY REPORT - approved counts")
print("=" * 64)
print(summary.to_string(index=False))
print("-" * 64)
print(f"Development pool (train + validation) : "
      f"{len(dev_pool)} images, {dev_pool['PatientId'].nunique()} patients")
print(f"All splits combined                   : "
      f"{summary['rows'].sum()} images, {len(all_patients)} patients")
print(f"Duplicate Image records removed       : {dup_records}")
print("-" * 64)
print(f"rows == unique images in every split  : {rows_equal_images}")
print(f"Image overlap, all split pairs        : "
      f"{len(train_images & valid_images) + len(train_images & test_images) + len(valid_images & test_images)}")
print(f"PatientId overlap, all split pairs    : "
      f"{len(train_patients & valid_patients) + len(train_patients & test_patients) + len(valid_patients & test_patients)}")
print("=" * 64)

INTEGRITY REPORT - approved counts
           split  rows  unique_images  unique_patients
     train_clean   795            795              744
     valid_clean   207            207              186
test (untouched)   420            420              389
----------------------------------------------------------------
Development pool (train + validation) : 1002 images, 930 patients
All splits combined                   : 1422 images, 1319 patients
Duplicate Image records removed       : 198
----------------------------------------------------------------
rows == unique images in every split  : True
Image overlap, all split pairs        : 0
PatientId overlap, all split pairs    : 0


## 6. Conclusion

- **Raw course data remains unchanged** under `data/raw/xray/nih/` and `data_dl/`.
- The development pool is **deduplicated by `Image`** before splitting, so every
  row count is a true image count. 198 duplicate records were removed.
- The split is **leakage-safe at both levels**: no `Image` and no `PatientId` is
  shared between train, validation and test.
- Rare classes, **especially Hernia**, remain highly imbalanced.
- **Class weights must be calculated later from `train_clean` only**
  (never from validation or test).
- The **official test set remains untouched** until final evaluation.

### Approved counts

| Split | Images | Patients |
|---|---|---|
| `train_clean.csv` | 795 | 744 |
| `valid_clean.csv` | 207 | 186 |
| `test.csv` (untouched) | 420 | 389 |
| **Development pool** | **1002** | **930** |

> Any previously reported figure of **948 / 252 / 1200** was a *row* count that
> included 198 duplicate records. Those numbers are superseded and must not be
> reused in the app or the report.

**Metrics produced before this fix are invalid** — 19.8% of development images
carried double weight. Baseline and fine-tuned classification training must be
re-run against these corrected CSVs.